# Exploratory Data Analysis — Energy Consumption Dataset

This notebook uses the cleaned dataset produced in Task 1. It calculates descriptive statistics, explores patterns, generates charts, and flags potential anomalies. The analysis describes associations; it does not establish causation.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Run from the project root. If needed, upload cleaned_dataset.csv in Colab.
data_path = Path("cleaned_dataset.csv")
if not data_path.exists():
    try:
        from google.colab import files
        uploaded = files.upload()
        data_path = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError("Place cleaned_dataset.csv in the same folder as this notebook.")

df = pd.read_csv(data_path)
df["Timestamp"] = pd.to_datetime(df["Timestamp"], format="%d-%m-%Y %H:%M")
df["Hour"] = df["Timestamp"].dt.hour
print("Shape:", df.shape)
print("Time range:", df["Timestamp"].min(), "to", df["Timestamp"].max())
display(df.head())

## 1. Dataset quality and descriptive statistics

In [ ]:
print("Missing values:", df.isna().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
numeric_cols = ["Temperature", "Humidity", "SquareFootage", "Occupancy",
                "RenewableEnergy", "EnergyConsumption"]
display(df[numeric_cols].describe().T.round(2))

## 2. Distribution of energy consumption

In [ ]:
plt.figure(figsize=(8,5))
plt.hist(df["EnergyConsumption"], bins=25, edgecolor="black")
plt.title("Distribution of Energy Consumption")
plt.xlabel("Energy Consumption")
plt.ylabel("Number of records")
plt.tight_layout()
plt.show()

## 3. Relationships and correlations

In [ ]:
corr = df[numeric_cols].corr()
display(corr.round(3))
plt.figure(figsize=(8,6))
plt.imshow(corr, vmin=-1, vmax=1, aspect="auto")
plt.colorbar(label="Pearson correlation")
plt.xticks(range(len(numeric_cols)), numeric_cols, rotation=45, ha="right")
plt.yticks(range(len(numeric_cols)), numeric_cols)
for i in range(len(numeric_cols)):
    for j in range(len(numeric_cols)):
        plt.text(j, i, f"{corr.iloc[i,j]:.2f}", ha="center", va="center", fontsize=8)
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()

plt.figure(figsize=(8,5))
plt.scatter(df["Temperature"], df["EnergyConsumption"], alpha=0.6)
m, b = np.polyfit(df["Temperature"], df["EnergyConsumption"], 1)
x = np.linspace(df["Temperature"].min(), df["Temperature"].max(), 100)
plt.plot(x, m*x+b)
plt.xlabel("Temperature")
plt.ylabel("Energy Consumption")
plt.title("Temperature vs Energy Consumption")
plt.tight_layout()
plt.show()

## 4. Group comparisons

In [ ]:
# HVAC status
display(df.groupby("HVACUsage")["EnergyConsumption"].agg(["count","mean","median"]).round(2))
df.groupby("HVACUsage")["EnergyConsumption"].mean().plot(kind="bar", title="Average Energy Consumption by HVAC Status", ylabel="Average Energy Consumption")
plt.tight_layout()
plt.show()

# Day of week
day_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]
day_means = df.groupby("DayOfWeek")["EnergyConsumption"].mean().reindex(day_order)
display(day_means.round(2))
day_means.plot(kind="bar", title="Average Energy Consumption by Day of Week", ylabel="Average Energy Consumption")
plt.tight_layout()
plt.show()

# Occupancy quantiles
df["OccupancyBand"] = pd.qcut(df["Occupancy"], 4, labels=["Lowest occupancy","Lower-middle","Upper-middle","Highest occupancy"])
occ_summary = df.groupby("OccupancyBand", observed=True)["EnergyConsumption"].agg(["count","mean","median"])
display(occ_summary.round(2))
occ_summary["mean"].plot(kind="bar", title="Average Energy Consumption by Occupancy Group", ylabel="Average Energy Consumption")
plt.tight_layout()
plt.show()

## 5. Hour-of-day pattern

In [ ]:
hourly = df.groupby(df["Timestamp"].dt.hour)["EnergyConsumption"].mean()
display(hourly.round(2))
hourly.plot(marker="o", figsize=(9,5), title="Average Energy Consumption by Hour", xlabel="Hour of day", ylabel="Average Energy Consumption")
plt.xticks(range(0,24,2))
plt.tight_layout()
plt.show()

## 6. Potential anomalies using the IQR rule

In [ ]:
q1, q3 = df["EnergyConsumption"].quantile([0.25, 0.75])
iqr = q3-q1
lower_fence, upper_fence = q1-1.5*iqr, q3+1.5*iqr
outliers = df[(df["EnergyConsumption"] < lower_fence) | (df["EnergyConsumption"] > upper_fence)]
print("Lower fence:", round(lower_fence, 2), "Upper fence:", round(upper_fence, 2))
print("Flagged records:", len(outliers))
display(outliers[["Timestamp","EnergyConsumption","Temperature","Occupancy","HVACUsage","LightingUsage"]])
# A statistical outlier is not automatically an error. Review before modifying.

## 7. Summary of key findings\n\nUse the tables and charts above to discuss at least five insights. Explain that correlation and group differences are associations, not proof of cause and effect.